# Agentic Commerce Analyst — architecture walkthrough

This notebook teaches the **existing repository implementation** visually before inspecting code. It does not create a notebook-only agent, retrain models, or claim there are multiple autonomous agents. The actual design is one LangGraph supervisor that routes to bounded services and tools.

## Diagram 1 — Streamlit → FastAPI → agent → response

```mermaid
flowchart TD
 U[User business question] --> UI[Streamlit analyst page]
 UI --> API[FastAPI POST /api/v1/agent/ask]
 API --> G[Guardrails: length, injection, PII, clarification]
 G -->|accepted| S[Single LangGraph supervisor]
 G -->|blocked / clarify| R[Controlled response]
 S --> T[Allowlisted existing service tool]
 T --> V[Validate tool result]
 V --> D{More evidence needed?}
 D -->|yes, based on observations| T
 D -->|no| Y[Synthesize and output guard]
 Y --> API --> UI
```

**What/why:** the API validates the request; the graph rejects unsafe or vague input before choosing tools; validated evidence is synthesized and returned. This keeps UI concerns and ML algorithms out of the supervisor.

| Architecture component | Real implementation | Role |
|---|---|---|
| Streamlit UI | `streamlit/pages/13_Agentic_Analyst.py` | Plain-language form, answer and concise trace |
| API client | `streamlit/utils/api_client.py` | Calls backend |
| FastAPI | `backend/app/main.py`, `backend/app/api/agent.py` | Registers and serves `/api/v1/agent/ask` |
| Request/response schemas | `backend/app/schemas/contracts.py` | Pydantic boundary |
| Guardrails | `backend/app/agents/supervisor.py` (`_guard`, `_output_guard`), `backend/app/guardrails/` | Lightweight input checks and output/source validation |
| Supervisor graph | `backend/app/agents/supervisor.py` | State transitions, bounded execution and synthesis |
| Presentation sanitizer | `backend/app/agents/presentation.py` | API evidence fields safe for the UI |

## Diagram 2 — Supervisor routing to existing capabilities

```mermaid
flowchart TB
 Q[Question + request state] --> P[Understand intent / allowed objectives]
 P --> C[Customer capabilities<br/>cohort, RFM, CLV, churn, recommendation]
 P --> A[Analytics capabilities<br/>KPIs, forecast, anomaly]
 P --> K[Knowledge capability<br/>RAG retrieval]
 C --> X[Registered deterministic tools]
 A --> X
 K --> X
 X --> O[Observe actual validated result]
 O --> D[Selector uses objectives + prior results]
 D -->|eligible next tool| X
 D -->|complete / no safe action| F[Final synthesis]
```

These are **capability groupings**, not three autonomous agents. The actual implementation has one supervisor and a registry of existing service adapters.

| Capability / component | Real implementation |
|---|---|
| Intent/objective planning | `backend/app/agents/planner.py` |
| State | `backend/app/agents/state.py` |
| Result-aware selection | `backend/app/agents/selector.py` and `router.py` |
| Tool allowlist, input/output contracts | `backend/app/tools/registry.py` |
| Customer features | `backend/app/tools/cohort_tool.py`, `rfm_tool.py`, `clv_tool.py`, `churn_tool.py` |
| Proxy fallbacks | `backend/app/tools/historical_value_tool.py`, `rfm_risk_tool.py` |
| Analytics | `backend/app/tools/analytics_tool.py`, `anomaly_tool.py`, `forecast_tool.py` |
| Recommendation / sentiment / RAG | `backend/app/tools/recommendation_tool.py`, `sentiment_tool.py`, `rag_tool.py` |
| Existing business services | `backend/app/services/commerce.py`, `rag_service.py`; shared features/services under `src/` |

## Diagram 3 — Mixed customer question uses results to decide what comes next

```mermaid
flowchart TD
 U[High-value customers likely to churn; recommend products?] --> C[customer_cohort_tool]
 C --> OC[Validate customer IDs + RFM features]
 OC --> D1{Actual cohort rows available?}
 D1 -->|yes| R[customer_rfm_tool]
 D1 -->|no| STOP[Explain missing evidence]
 R --> O1[Observe feature rows]
 O1 --> D2{CLV objective unresolved?}
 D2 -->|yes| CLV[clv_prediction_tool]
 CLV -->|model contract fails| HV[historical_value_tool fallback]
 CLV -->|valid output| CH[churn_prediction_tool]
 HV --> CH
 CH -->|model contract fails| RR[rfm_risk_tool fallback]
 CH -->|valid output| REC[recommendation_tool]
 RR --> REC
 REC --> SYN[Synthesize with proxy/model labels]
```

The route is a dynamic observe → decide → act loop. The exact next call is selected from the current objectives and real prior results. Registered fallback tools are only used after primary failure; they are not extra autonomous agents.

| Graph stage | Code |
|---|---|
| Guard, understand, select, execute, observe, reason | `backend/app/agents/supervisor.py` |
| Candidate action and result-dependent inputs | `backend/app/agents/router.py`, `selector.py` |
| Churn/CLV output contracts | `backend/app/tools/registry.py` |
| Cohort/RFM/churn/CLV/recommendation handlers | `backend/app/tools/` |
| Runtime proof | `backend/tests/test_agent_runtime.py` |

## Diagram 4 — Structured data vs documents vs unavailable evidence

```mermaid
flowchart TD
 Q[Business question] --> K{What evidence does it require?}
 K -->|Commerce rows / model features / KPI| T[Relevant structured tool]
 K -->|Indexed project documents| RAG[rag_search]
 K -->|Both| MIX[Tools + RAG when documents add evidence]
 K -->|Not present in data or knowledge base| MISS[State limitation; no invented metric]
 T --> V[Validate actual result]
 RAG --> RV[Require non-empty evidence + sources]
 MIX --> V
 MISS --> SAFE[Offer clearly labeled general guidance only if suitable]
 V --> A[Project-data-backed synthesis]
 RV --> B[Knowledge-base-backed synthesis]
```

A registry tool can only answer from its implemented inputs and services. RAG is for indexed documents and is not a substitute for missing competitor, marketing-channel, currency, or support-preference data. General guidance must be labeled as such.

| Existing data / knowledge component | Real path |
|---|---|
| Prepared commerce data + prediction/recommendation services | `backend/app/services/commerce.py`, `backend/app/core/config.py` |
| Customer feature engineering | `src/features/customer.py` |
| Retrieval index/service | `backend/app/services/rag_service.py`, `src/services/retrieval.py` |
| RAG tool adapter | `backend/app/tools/rag_tool.py` |
| Output source checks | `backend/app/agents/supervisor.py` |

## Decision table (conceptual, not a hard-coded restriction)

| Intent | Capability group |
|---|---|
| Customer value, churn, CLV, segmentation, RFM | Customer |
| Forecast, revenue, KPIs, anomaly | Analytics |
| Product recommendation | Customer recommendation capability |
| Knowledge/document question | Knowledge retrieval |
| Mixed business question | Multiple capabilities, selected from evidence |
| Missing-data question | Limitation; optionally sourced RAG/general guidance |

The planner proposes allowed objectives; the result-aware selector determines one eligible next tool at a time. Keyword rules are the deterministic fallback when the local LLM planner is unavailable or disabled.

## Repository audit: keep, modify, or avoid duplication

| Existing files | Current role | New architecture role | Decision |
|---|---|---|---|
| `backend/app/services/commerce.py` | Loads prepared data and model/service operations | Existing domain workers | Keep; adapters call it |
| `src/features/customer.py`, `src/services/` | Shared feature/recommendation/sentiment/retrieval logic | Reusable service layer | Keep and reuse |
| `backend/app/agents/planner.py`, `router.py`, `selector.py` | Intent objectives and one next eligible action | Supervisor planning/selection | Keep and clarify |
| `backend/app/agents/supervisor.py`, `state.py` | LangGraph stages and per-request state | Central orchestration | Modify for readability only where needed |
| `backend/app/tools/` | Typed adapters and registry | Existing worker interfaces | Keep; do not duplicate models |
| `backend/app/guardrails/` | Pattern checks and policy | Input/output safety | Keep; describe as lightweight |
| `backend/app/evaluation/`, `backend/tests/` | Execution evaluation and regression cases | Runtime evidence | Keep and extend only for gaps |
| `streamlit/pages/13_Agentic_Analyst.py` | Existing analyst UI | User interaction/evidence view | Modify presentation |
| `notebooks/11_Agentic_AI_Orchestration.ipynb` | Agent walkthrough | Visual architecture and runnable examples | Modify teaching flow |
| Other notebooks and model artifacts | Training/analysis artifacts | Existing domain implementation | Keep unchanged |
| `backend/app/` vs `src/` | API runtime vs shared services | Existing project split | No duplicate agent tree |

## Folder view

```text
backend/app/
├── agents/       # state, planner, selector, router, supervisor, presentation
├── api/          # FastAPI routes including agent endpoint
├── guardrails/   # lightweight injection/PII/policy checks
├── schemas/      # request and response contracts
├── services/     # existing commerce and RAG services
├── tools/        # thin tool adapters and registry
├── evaluation/   # golden cases and execution evaluation
└── monitoring/   # runtime metrics
streamlit/pages/  # user-facing analyst page
notebooks/        # visual/code walkthrough
models/, data/    # existing artifacts and prepared data
```

## Notebook setup

The following cells import the project’s actual graph and adapters. Run from the repository or its `notebooks/` folder. Live data examples remain opt-in because they execute project tools against local data and model artifacts.


## Execution examples and verification matrix

The next cell calls the real `run_agent` graph. It is opt-in because requests use the repository's prepared data, local saved models, and retrieval index. Its output reports the **observed tools and final task status**, without printing row-level customer records.

| Question | Expected capability route | Runtime/test evidence |
|---|---|---|
| Which customers are likely to churn? | cohort → RFM → churn (or registered RFM proxy) | Combined flow and `test_churn_retry_then_executes_real_rfm_fallback` |
| Which customers have the highest CLV? | cohort → RFM → CLV (or historical-value proxy) | Combined flow; CLV output contract tests |
| Show customer RFM segments | cohort → RFM | `test_customer_cohort_uses_existing_feature_builder` |
| Which products should we recommend? | cohort → recommendation | Combined flow verifies IDs originate in tool results |
| Forecast demand next week | forecast | `test_existing_forecast_and_faiss_rag_services_have_tool_adapters` |
| What are our primary KPIs? | business analytics | Golden cases and analytics output contract |
| Search the knowledge base for delivery policy | RAG retrieval with evidence/source checks | RAG adapter and empty-evidence tests |
| High-value customers likely to churn; recommend products? | cohort → RFM → CLV → churn → recommendation | `test_tool_result_drives_next_tool_and_inputs` |
| Festival coming; increase revenue? | cohort → recommendation → KPI → forecast | Capability classification test; optional live scenario below |
| How do we compare against competitors? | check RAG; if unsupported, state limitation | RAG empty-evidence behavior; do not invent competitor metrics |
| Ignore instructions / share email PII | guardrail → controlled block | `test_guardrail_blocks_injection_and_pii_before_tools` |
| Churn service fails | bounded retry → actual RFM risk fallback | `test_churn_retry_then_executes_real_rfm_fallback` |

Expected routes above are capability-level expectations. Actual tool names and success/fallback status are printed by the live cell; tool contracts or missing local evidence can produce a safe partial result.


## 3. Resolve the project and import the production implementation

Run from the repository root or this `notebooks/` directory. The code resolves the repository by its backend marker instead of relying on a Windows-specific path.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current=(start or Path.cwd()).resolve()
    for path in (current,*current.parents):
        if (path/"backend/app/agents/supervisor.py").exists():
            return path
    raise FileNotFoundError("AI-Commerce-Analytics-Platform root not found")

PROJECT_ROOT=find_project_root()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
from backend.app.agents.supervisor import agent_graph, run_agent, MAX_ITERATIONS
from backend.app.agents.state import AgentState
from backend.app.agents.planner import understand_request
from backend.app.agents.selector import select_next_action
from backend.app.tools.registry import TOOL_REGISTRY, validate_inputs, validate_output
from backend.app.evaluation.golden_dataset import GOLDEN_DATASET
print("Project:",PROJECT_ROOT)
print("Allowlisted tools:",list(TOOL_REGISTRY))

## 4. Actual project integration map

| Runtime concern | Implementation |
|---|---|
| State and graph | `backend/app/agents/state.py`, `supervisor.py` |
| Intent and next-tool selection | `backend/app/agents/planner.py`, `router.py`, `selector.py` |
| Observation, synthesis, output checks | `backend/app/agents/response.py` |
| Privacy-conscious API evidence formatting | `backend/app/agents/presentation.py` |
| Cohort and bounded service adapters | `backend/app/tools/` |
| Input injection/PII and output checks | `backend/app/guardrails/`, graph output guard |
| Runtime evaluation and goldens | `backend/app/evaluation/` |
| FastAPI | `POST /api/v1/agent/ask` in `backend/app/api/agent.py` |
| Streamlit | `streamlit/pages/13_Agentic_Analyst.py` |
| Docker | LangGraph dependency in `backend/requirements.txt`, planner config in Compose |

The organization stays under `backend/app` because that is where the existing API service layer lives. Duplicate agent implementations under `src/` are intentionally avoided.

## Repository audit and reuse decisions

| Existing area / files | Existing responsibility | New architecture role | Decision |
|---|---|---|---|
| `streamlit/pages/01_Home.py`, `02_Customer_Analytics.py` | Platform overview and customer analytics | Existing UI/API clients | Keep |
| `streamlit/pages/03_Churn_Prediction.py`, `04_CLV_Prediction.py` | Existing churn and value interfaces | Existing model capabilities | Keep |
| `streamlit/pages/05_Delivery_Delay.py`, `06_Product_Recommendation.py` | Delivery and recommendation experiences | Existing service workers | Keep |
| `streamlit/pages/07_Sentiment_Analysis.py`, `08_Demand_Forecasting.py` | Sentiment and demand forecast | Existing ML/service workers | Keep |
| `streamlit/pages/09_RAG_Chatbot.py`, `11_RAG_Operations.py` | Knowledge chat and retrieval monitoring | Existing knowledge interface/ops | Keep; analyst uses RAG only when relevant |
| `streamlit/pages/10_About.py`, `12_Monitoring.py` | Project and platform monitoring | Existing platform UI | Keep |
| `streamlit/pages/13_Agentic_Analyst.py` | Analyst question UI | Plain-language agent interface | Modify to show concise answer and sanitized trace |
| `backend/app/api/routes.py` | `/health`, customer metrics, churn/CLV/delivery predictions, recommendations, sentiment, forecast, chat, RAG metrics | Existing API and service boundaries | Keep |
| `backend/app/api/agent.py`, `main.py` | New `/api/v1/agent/ask` route registered with existing API | Agent API boundary | Keep |
| `backend/app/services/commerce.py` | Prepared data, saved artifacts, KPI/model/forecast/recommendation/sentiment services | Domain workers behind adapters | Reuse; do not reimplement algorithms |
| `backend/app/services/rag_service.py`, `src/services/retrieval.py` | Indexed document retrieval and RAG services | Knowledge worker | Reuse through `rag_tool.py` |
| `src/features/customer.py`, `src/services/recommendations.py`, `sentiment.py` | Shared customer features and existing recommendation/sentiment logic | Feature/service layer | Reuse |
| `backend/app/tools/*.py`, `registry.py` | Thin handlers, schemas and result validation | Tool registry/execution boundary | Keep and extend only for missing routes |
| `backend/app/agents/{planner,router,selector,state,supervisor}.py` | Request understanding and state-aware tool loop | Single supervisor agent | Refactor for clarity without changing graph behavior |
| `backend/app/guardrails/`, `evaluation/`, `monitoring/` | Pattern checks, runtime scoring, metrics | Safety/evaluation/observability | Keep; label lightweight checks honestly |
| `notebooks/01–10`, `models/`, `data/`, `rag_ops/` | Existing analyses, model artifacts, prepared data, RAG operations | Existing project assets | Keep; no duplicate project or model pipeline |

The audit found no autonomous customer, analytics, or knowledge agent modules. Those names in diagrams describe **capability groups**. The implemented runtime remains one supervisor calling deterministic registered tools.


## 5. Agent state

The typed state in `backend/app/agents/state.py` carries the query, objectives, current tool, validated call/results, observations, next action, retry/fallback counts, memory context, guardrail result, anomalies, sources, final response, evaluation, and safe trace. Results are available to subsequent decisions through `tool_results` and `observations`; the runtime does not execute a prebuilt queue.

In [ ]:
print(AgentState.__annotations__.keys())
print("Graph nodes:",list(agent_graph.get_graph().nodes))
print("Maximum tool attempts/iterations:",MAX_ITERATIONS)

In [ ]:
RUN_SCENARIO_EXAMPLES=False
SCENARIO_QUESTIONS=[
    "Which customers are likely to churn?",
    "Which customers have the highest CLV?",
    "Show customer RFM segments",
    "Which products should we recommend?",
    "Forecast demand for next week",
    "What are our primary KPIs?",
    "Search the knowledge base for delivery policy",
    "Which high-value customers are likely to churn and what products should we recommend?",
    "A festival is coming. How can we increase revenue?",
    "How do we compare against competitors?",
]
if RUN_SCENARIO_EXAMPLES:
    for scenario in SCENARIO_QUESTIONS:
        observed=run_agent(scenario,planner_mode="rules",selector_mode="rules")
        print({"question":scenario,"tools_called":observed["tools_used"],
               "status":"completed" if observed.get("evaluation",{}).get("task_success") else "partial_or_limited",
               "fallback_used":observed.get("fallback_used"),"grounded":observed.get("grounded"),
               "latency_ms":observed.get("latency_ms")})
else:
    print("Scenario calls are opt-in. Set RUN_SCENARIO_EXAMPLES=True to capture actual local tool routes.")


## 6. Registry contracts, input validation, and output validation

Each `ToolSpec` has a registered name, description, JSON-like schema, handler, output contract, retry limit, optional fallback tool, and risk level. Runtime validation checks required and extra inputs, types, ranges, enums, business constraints, output fields, and selected domain rules. Fallback tools are registered but marked non-selectable so the LLM cannot invoke them as ordinary goals.

In [ ]:
for name,spec in TOOL_REGISTRY.items():
    print(name,{"description":spec.description,"schema":spec.schema,"max_retries":spec.max_retries,
                "fallback_tool":spec.fallback_tool,"risk_level":spec.risk_level,"selectable":spec.selectable})

## 7. Planning is request understanding, not a task queue

`understand_request` returns an intent and allowed objectives. It does not ask the LLM for `[tool1, tool2, tool3]`. Invalid or unavailable local Ollama planning falls back to deterministic intent recognition. Obvious user requirements are merged back into the validated objective set so a planner cannot silently discard them.

In [ ]:
intent,objectives,planner_mode,llm_calls=understand_request(
    "Which high-value customers may churn and what could we recommend?",mode="rules")
print({"intent":intent,"objectives":objectives,"planner_mode":planner_mode,"llm_calls":llm_calls})

## 8. Result-aware tool selection

`select_next_action` uses the goals and actual validated results in state to build only feasible candidate actions. The LLM receives registry descriptions/schemas, eligible candidate names, and safe observation summaries; it never receives retrieved document bodies, raw customer feature rows, or tool arguments. The host constructs arguments from the query and results, then validates them. The deterministic selector is the fallback when Ollama fails.

## 9. LangGraph is the real observe → decide → act loop

The compiled graph routes one action at a time: select → execute → validate → observe → reason → select again or synthesize. Conditional edges handle retry, fallback, invalid output, blocked input, stop conditions, and the maximum iteration limit. The graph’s recursion limit is a separate secondary guard, not the agent iteration counter.

In [ ]:
print(agent_graph.get_graph().draw_mermaid())

## 10. Customer cohort and model feature contracts

`customer_cohort_tool` uses `src.features.customer.build_customer_features` over the existing master data. `high_value` uses historical monetary value at or above the 75th percentile; `at_risk` is descriptive recency >180 days. Cohort rows include the actual derived fields. Churn/CLV tools compare those columns with the saved estimator’s `feature_names_in_`; they never invent missing model inputs. A mismatch becomes a recorded tool failure.

In [ ]:
cohort_spec=TOOL_REGISTRY["customer_cohort_tool"]
print(cohort_spec.schema)
print("Churn estimator path:",PROJECT_ROOT/"models/customer_churn_model.joblib")
print("CLV estimator path:",PROJECT_ROOT/"models/customer_clv_model.joblib")

## 11. Execute a live query through the same backend graph

Set `RUN_LIVE_DEMOS=True` only after project dependencies, processed data, models, and any required FAISS index are available. Each query invokes the real `run_agent`; unavailable artifacts appear as errors/fallbacks. No demo result is fabricated.

In [ ]:
RUN_LIVE_DEMOS=False
if RUN_LIVE_DEMOS:
    revenue_result=run_agent("What is the total revenue?",planner_mode="rules",selector_mode="rules")
    display(revenue_result)
else:
    print("Live calls are disabled. Set RUN_LIVE_DEMOS=True to execute against this repository's services.")

## 12. Conditional multi-tool investigation

For “Which high-value customers are likely to churn and what products should we recommend?”, the deterministic path starts with a high-value cohort, then RFM. Once those actual customer rows exist, CLV and churn become eligible; churn receives the CLV-enriched rows if CLV succeeded. Recommendations use identifiers returned by previous results. If the CLV/churn tool fails, a registered fallback can run. The exact path can change when results, feature contracts, or the LLM selector decision differ.

In [ ]:
if RUN_LIVE_DEMOS:
    investigation=run_agent(
        "Which high-value customers are likely to churn and what products should we recommend?",
        planner_mode="rules",selector_mode="rules")
    display({key:investigation[key] for key in ("answer","tools_used","fallback_used","errors","evaluation","trace")})

## 13. Retry and real fallback

Transient failures receive at most two retries (per-tool limits also apply). Input/schema errors are not retried. After churn failure, the graph actually executes `rfm_risk_tool`; its output is explicitly marked as a recency heuristic and never presented as a trained churn probability. CLV failure can invoke `historical_value_tool`. RAG has no misleading string fallback: retrieval failure produces a controlled partial/unsupported answer.

In [ ]:
print("Churn fallback:",TOOL_REGISTRY["churn_prediction_tool"].fallback_tool)
print("CLV fallback:",TOOL_REGISTRY["clv_prediction_tool"].fallback_tool)
print("Fallback execution and retry assertions live in backend/tests/test_agent_runtime.py")

## 14. Iteration bounds and execution anomalies

Each tool attempt increments `iteration_count`; retries consume the same global budget. Reaching the limit stops the graph, records an anomaly, and preserves partial results. The runtime also detects repeated identical tool/result signatures, A/B tool oscillation, excessive retries/fallbacks, empty retrieval, and tool latency spikes. Business metric anomalies remain separate and use the project’s MAD-based detector.

## 15. Response synthesis and grounding

The graph has separate synthesis, output-guard, and evaluation nodes. The synthesizer turns successful typed results into a concise business summary; it does not JSON-dump the entire state and does not reveal chain-of-thought. The output guard redacts common email/phone patterns and checks retrieved-source provenance. `grounded` is a categorical evidence status, not a fabricated score.

## 16. Memory scope

Current memory is request-scoped LangGraph state and a compact list of completed tools. There is no SQLite, JSON, or other long-term user memory. Do not describe this implementation as cross-session memory.

## 17. Single-agent vs multi-agent

This implementation is **single-agent orchestration over deterministic tools**. `supervisor.py` is a graph/supervisor function, not a collection of autonomous customer, analytics, and knowledge agents. There is no multi-agent delegation.

## 18. Input and retrieval guardrails

Input rules enforce non-empty/length limits, block a few common prompt-injection patterns, block email/phone-like PII, and request clarification for a few obviously vague prompts. This is a demo detector, not a complete classifier. Retrieved documents are treated as untrusted data: the planner never sees retrieval output, and the selector sees only source/count summaries rather than document text. No toxicity or misuse classifier is installed.

## 19. Output safety and unsupported claims

Tool outputs pass contract and domain checks before entering observations. The response is assembled from validated results; model probabilities, CLV numbers, and recommendations must originate from their registered tools. If evidence is absent or retrieval sources fail validation, the response is marked unknown/false instead of inventing grounding. Common contact details are redacted; broader PII detection is not implemented.

## 20. Runtime evaluation

`backend/app/evaluation/agent_eval.py` executes requests and scores intent, first-tool selection, safe argument summaries, task completion, plan/tool order, retries, fallback rate/correctness where expected, tool calls, iterations, latency, failure, and grounding. The notebook golden set includes analytics, RFM, churn, CLV, forecast, recommendation, sentiment, RAG, conditional investigation, ambiguity, injection, PII, and loop-limit cases. Separate tests force invalid tool calls, retry/fallback, and empty RAG.

In [ ]:
from backend.app.evaluation.agent_eval import evaluate_runtime_cases,benchmark_agent_versions
print("Golden cases:",len(GOLDEN_DATASET))
print("Evaluation API:",evaluate_runtime_cases.__name__,benchmark_agent_versions.__name__)

## 21. Version benchmark

V1 uses deterministic request understanding and rules selection; V2 uses the schema-checked LLM planner with rules selection; V3 uses the LLM planner plus the dynamic result-aware selector. All execute the same golden requests and score actual tool sequence, argument validity, task completion, result, retries, fallbacks, guardrail, latency, and answer. V2/V3 only represent real LLM runs when Ollama is available; response modes show deterministic fallback when it is not.

In [ ]:
RUN_FULL_BENCHMARK=False
if RUN_FULL_BENCHMARK:
    versions=benchmark_agent_versions(run_agent)
    for version,result in versions.items():
        print(version,result["metrics"])
else:
    print("Benchmark disabled: it makes repeated real service/LLM calls. Enable when ready.")

## 22. Execution tests

The backend tests prove a prior CLV result is passed into churn selection, test execution order, retries and actual RFM fallback, schema/output rejection, guardrails, PII redaction, empty RAG, repeated calls, iteration limits, and benchmark-mode separation.

In [ ]:
# From the repository root, in the project environment after installing backend/requirements.txt:
# python -m pytest backend/tests/test_agent_runtime.py

## 23. Observability and safe trace

The API returns a safe execution trace with tool, status, redacted input summary, latency, attempt, fallback relationship, and sources. It does not return raw customer feature inputs. Logs omit full questions/feature rows. Prometheus tracks request outcome, tool calls/failures/latency, retries, fallbacks, iterations, LLM latency, and execution anomalies with low-cardinality labels. Token usage and cost are `None` because the current Ollama client path does not expose reliable accounting.

## 24. Streamlit integration

The **Agentic Analyst** page calls `POST /api/v1/agent/ask`, accepts a question, optional platform customer ID, and optional review text, then displays the summary, status, tool sequence, sources, errors/fallbacks, safe trace, and evaluation. It does not display hidden reasoning or raw tool arguments.

## 25. FastAPI integration

The API route is registered in `backend/app/main.py`. Request/response schemas live in `backend/app/schemas/contracts.py`; execution is synchronous in FastAPI’s worker pool. The existing project has no API authentication, so customer-scoped endpoints must not be exposed publicly without adding authentication and authorization.

## 26. Docker and local configuration

`backend/requirements.txt` includes LangGraph; `docker-compose.yml` passes `AGENT_PLANNER_ENABLED` and the planner timeout to the backend. Set `AGENT_PLANNER_ENABLED=false` for deterministic-only behavior. The existing Ollama service/host networking and project artifacts are reused; no new database or agent microservice is introduced.

## 27. Latency, cost, and tokens

End-to-end and tool latency are measured, and the current planner/selector LLM latency is counted when available. More decisions can add Ollama latency. Early finishing, bounded iterations, no unnecessary RAG, and result caching are possible optimizations. This implementation does not calculate token usage or currency cost; fields remain null until the runtime exposes reliable values.

## 28. KV cache

KV cache is an inference-engine optimization. The application can benefit from it but does not directly implement LLM internal KV-cache management.

## 29. End-to-end scenarios

With live calls enabled, try simple revenue analytics; high-risk cohort → RFM → churn; high-value cohort → RFM → CLV → churn → recommendations; business anomaly → conditional RAG investigation only if flags exist; RAG with source validation; and sentiment with provided review text. Failure/retry/fallback, injection, PII, max iterations, empty retrieval, and repeated execution are covered by tests.

## 30. Architecture that matches runtime

```mermaid
flowchart TD
 U[User] --> G[Input guard and PII / injection checks] --> P[Understand intent and objectives]
 P --> S[Result-aware tool selector]
 S --> E[Allowlisted tool executor]
 E --> V[Input/output contract validation]
 V -->|valid| O[Observe result and update request memory]
 V -->|retryable and budget remains| R[Bounded retry] --> E
 V -->|retries exhausted| F[Registered fallback tool] --> E
 V -->|invalid / no fallback / budget exhausted| Y[Synthesize partial evidence]
 O --> D[Reason: next safe action or done]
 D -->|continue| S
 D -->|done / repeated-call anomaly| Y
 Y --> OG[Output PII and source guard]
 OG --> Q[Runtime evaluation]
 Q --> M[Structured trace and Prometheus]
 M --> END[Grounded response]
 X[RAG result] -. untrusted document data; selector receives only source/count .-> O
```

There are no autonomous specialist-agent branches and no persistent memory node.

## 31. Interview-ready summary

I added a LangGraph-based, single-agent orchestration layer over the commerce platform’s existing ML and RAG services. It understands user intent, selects one schema-validated tool at a time, observes and validates results, and selects the next action using state and previous outputs. Retries are bounded; churn/CLV fallbacks execute real registered heuristic tools and are labeled as proxies. The API returns grounded evidence and safe telemetry; memory is request-scoped. This is not a multi-agent or long-term-memory system, and toxicity/misuse classification is not implemented.